In [13]:
from build123d import *
from ocp_vscode import *
from math import sin, cos,tan, pi
set_port(3939)
set_defaults(reset_camera=Camera.CENTER, helper_scale=5)
import numpy as np

In [2]:
# Copying from https://github.com/gumyr/build123d/commit/7fb6e280f6142f03f7f8376531ffb7a15a071a89
# since that is not in the current release
from __future__ import annotations

from collections.abc import Iterable
from math import radians, tan
from scipy.spatial import ConvexHull

from build123d.build_common import LocationList, validate_inputs
from build123d.build_enums import Align, Mode
from build123d.build_part import BuildPart
from build123d.geometry import (
    Location,
    Plane,
    Rotation,
    RotationLike,
    Vector,
    VectorLike,
)
from build123d.topology import (
    Compound,
    Face,
    Part,
    ShapeList,
    Shell,
    Solid,
    Wire,
    tuplify,
)


class ConvexPolyhedron(BasePartObject):
    """Part Object: ConvexPolyhedron

    Create a convex solid from the convex hull of the provided points.

    Args:
        points (Iterable[VectorLike]): vertices of the polyhedron
        rotation (RotationLike, optional): angles to rotate about axes. Defaults to (0, 0, 0)
        align (Align | tuple[Align, Align, Align] | None, optional): align MIN, CENTER,
            or MAX of object. Defaults to Align.NONE
        mode (Mode, optional): combine mode. Defaults to Mode.ADD
    """

    _applies_to = [BuildPart._tag]

    def __init__(
        self,
        points: Iterable[VectorLike],
        rotation: RotationLike = (0, 0, 0),
        align: Align | tuple[Align, Align, Align] | None = Align.NONE,
        mode: Mode = Mode.ADD,
    ):
        context: BuildPart | None = BuildPart._get_context(self)
        validate_inputs(context, self)

        pnts: list[tuple] = [tuple(Vector(p)) for p in points]

        # Create a convex hull from the vertices
        convex_hull = ConvexHull(pnts).simplices.tolist()

        # Create faces from the vertex indices
        polyhedron_faces = []
        for face_vertex_indices in convex_hull:
            corner_vertices = [pnts[i] for i in face_vertex_indices]
            polyhedron_faces.append(Face(Wire.make_polygon(corner_vertices)))

        # Create the solid from the Faces
        polyhedron = Solid(Shell(polyhedron_faces)).clean()

        super().__init__(
            part=polyhedron, rotation=rotation, align=tuplify(align, 3), mode=mode
        )


In [33]:
PYRAMID_POINTS = [
    (0, 0, 0),
    (2, 0, 0),
    (0, 2, 0),
    (2, 2, 0),
    (1, 1, 2),
]

ConvexPolyhedron(PYRAMID_POINTS)

ConvexPolyhedron at 0x767b4bbd37f0, label(), #children(0)

In [34]:
num_sides = 20
cyc_radius = 20
spike_len = 2
num_rows = 6
num_spikes = 61
with BuildPart() as p:
    Cylinder(cyc_radius, 2*num_rows+1)
    fillet(p.edges(), radius=0.5)
    with Locations([(0, 0, 2*x) for x in np.arange(-num_rows/2, num_rows/2)]):
        with PolarLocations(cyc_radius-0.1, num_spikes):
            x = ConvexPolyhedron(
                PYRAMID_POINTS, (0, 90, 0), 
                align=(Align.MAX, Align.NONE, Align.NONE),
            )
    
show(p)

+


In [35]:
export_stl(p.part, "spiky_stim.stl")

True